# 강의 06 · 실습 2 — 패턴 2 라우팅 · (2-2) 빈칸 채우기 II

## 1. 문제상황

- 총무팀 헬프데스크에는 비품 교체, 휴가 결재선, 보안 규정 같은 문의가 한 창구로 섞여 들어옵니다.
- 담당자는 문의를 읽고 비품 담당·휴가 담당·보안 담당 중 누가 답해야 하는지 고른 뒤, 그 담당자에게 문의를 넘깁니다.
- 담당마다 안내 규칙이 달라서, 한 사람이 세 창구의 답을 다 쓰려고 하면 규칙이 섞여 답이 틀립니다.
- 문의가 늘어나면 누가 답할지 고르는 일에 시간이 그만큼 더 들어갑니다.

## 2. 문제와 목표

- **문제**: 문의가 어느 창구의 일인지 사람이 고르고, 창구마다 다른 안내 규칙도 사람이 기억해서 답합니다. 고르는 일과 답하는 일이 문의 수만큼 반복됩니다.
- **목표**: 문의를 입력하면 분류 노드가 비품·휴가·보안 중 한 창구를 고정된 형식으로 고르고, 조건부 엣지가 그 창구의 전담 노드로 보내, 전담 노드가 자기 규칙대로 답하는 처리 흐름을 만듭니다.
    - 분류 노드: 문의를 비품·휴가·보안 중 하나로 고르는 구조화 출력 노드입니다.
    - 고정된 형식: 창구 이름 하나만 담는 구조화 출력 규격(값은 셋 중 하나로 제한).
    - 전담 노드 셋: 비품·휴가·보안 창구가 하나씩, 자기 안내 규칙대로 답합니다.
- **목표 달성 여부의 판정 기준**: 비품 문의와 휴가 문의를 차례로 입력했을 때, 두 입력이 분류 노드 뒤에 서로 다른 전담 노드 하나만 거치고 나머지 전담 노드는 실행되지 않는 것을 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec06_ex02_s1_diagram.svg)

## 4. 단계별 요구사항

1. **상태와 분류 규격을 정의합니다.**
    - 문의(`question`), 분류 결과(`decision`), 담당 창구의 답(`answer`) 키 세 개를 가지는 상태를 선언합니다.
    - 분류 결과의 규격 `Route`는 `step` 필드 하나를 가지며, 값은 비품·휴가·보안 셋 중 하나로만 제한합니다.
2. **분류 노드를 만듭니다.**
    - classify 노드는 `Route` 규격을 건 모델을 불러 문의를 세 창구 중 하나로 배정하고, 그 값을 상태의 `decision` 키에 씁니다.
3. **창구 노드 세 개를 만듭니다.**
    - supply 노드는 비품 창구로서 신청 창구와 처리 기간을, leave 노드는 휴가 창구로서 결재선과 신청 기한을, security 노드는 보안 창구로서 사내 규정 근거를 들어 각각 두 문장으로 답해 상태의 `answer` 키에 씁니다.
4. **그래프에 노드를 등록합니다.**
    - 네 노드를 이름과 함께 그래프에 등록합니다.
5. **엣지를 연결합니다.**
    - START에서 classify로 가는 고정 엣지를 추가합니다.
    - classify 뒤에는 `decision` 값을 보고 supply·leave·security 중 하나를 고르는 조건부 엣지를 추가합니다.
    - 판단 함수는 갈 노드의 이름만 돌려주고 상태를 바꾸지 않습니다.
    - 세 창구 노드 뒤에는 각각 END를 고정 엣지로 연결합니다.
6. **그래프를 컴파일하고 실행합니다.**
    - 비품 문의와 휴가 문의를 차례대로 넣고, 노드가 하나 끝날 때마다 어느 노드가 상태의 어느 키를 채웠는지 화면에 출력한 뒤, 분류 결과와 답을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키와 분류 결과의 규격을 선언합니다 | `class DeskState(TypedDict)`, `class Route(BaseModel)` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `llm.with_structured_output(Route)`, `def classify(state) -> dict` | 2, 3 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 함수에 이름을 붙여 등록합니다 | `StateGraph(DeskState)`, `add_node` | 4 |
| ④ 엣지 연결 | 분류 결과에 따라 나뉘는 분기를 지정합니다 | `add_edge`, `add_conditional_edges` | 5 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 입력을 넣어 실행합니다 | `compile()`, `stream()` | 6 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
# 여기에 단계 0(라이브러리 불러오기, .env 읽기, 모델 준비)을 작성합니다.

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. 상태와 함께 분류 결과의 규격 `Route`도 여기서 선언합니다. `Literal`에 적힌 값 밖으로는 분류 결과가 나오지 않으므로, 어느 창구로 보낼지의 판단 기준이 스키마 한 곳에 모입니다.

In [ ]:
# 여기에 단계 ①(상태 정의와 분류 규격 Route 정의)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3)

- 노드는 상태를 인자로 받아 딕셔너리를 돌려주는 파이썬 함수입니다. 돌려준 딕셔너리가 상태의 해당 키를 덮습니다.
- 분류 노드는 `llm.with_structured_output(Route)`로 감싼 모델을 부릅니다. 돌아오는 값은 문자열이 아니라 `Route` 객체이며, 분류 노드는 그 값을 상태의 `decision` 키에 넣기만 합니다.
- 창구 노드들은 같은 문의를 받아 같은 `answer` 키에 씁니다. 다른 것은 시스템 프롬프트뿐입니다. 창구끼리는 서로를 부르지 않습니다. 어느 창구가 실행될지는 함수 밖의 조건부 엣지가 정합니다.

In [ ]:
# 여기에 단계 ②(분류 노드와 창구 노드 세 개 정의)를 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 4)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다. 여기서 붙인 이름은 뒤의 엣지 연결에서 그대로 쓰입니다.

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 5)

`add_edge`는 고정된 순서로 연결합니다. `add_conditional_edges`는 판단 함수가 돌려준 이름으로 다음 노드가 나뉘는 분기를 추가합니다. 판단 함수 `route_decision`은 상태의 `decision` 키만 보고 갈 곳의 노드 이름을 돌려주며, 상태를 바꾸지 않습니다. 세 번째 인자는 갈 수 있는 노드 이름의 목록입니다.

In [ ]:
# 여기에 단계 ④(판단 함수와 엣지 연결)를 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 6)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `stream`은 노드가 하나 끝날 때마다 그 노드가 바꾼 키를 내보냅니다. 아래에서는 비품 문의와 휴가 문의를 차례대로 넣습니다.

In [ ]:
# 여기에 단계 ⑤(컴파일과 실행)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 1번 문의(의자 교체)에서는 `classify` 뒤에 `supply` 노드 한 줄만 출력됩니다. `decision` 값이 비품이므로 조건부 엣지가 supply를 골랐고, leave와 security는 실행되지 않습니다.
2. 2번 문의(연차 결재선)에서는 `classify` 뒤에 `leave` 노드 한 줄만 출력됩니다. 같은 그래프가 입력에 따라 다른 창구를 고릅니다.
3. 두 문의의 `decision` 값은 `Route`의 `Literal`에 적힌 세 값 중 하나입니다. 세 값 밖의 문자열은 나오지 않습니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다. `KeyError`가 나면 판단 함수의 딕셔너리 키와 `Literal`의 값이 같은지 단계 ①과 단계 ④를 대조합니다.